# 01 – Gemini Prompt Test

Goal:
Test Google Gemini's ability to answer research-style questions
with structured, clear, and controlled output.

This notebook is used to validate prompt engineering
before backend integration.


In [39]:
import sys
print(sys.executable)
#!pip install -q google-generativeai

c:\Users\pc\Desktop\ai_research_copilot\backend\venv\Scripts\python.exe


In [1]:
import os
from dotenv import load_dotenv
import google.generativeai as genai
from datetime import datetime
import json 


load_dotenv()
api_key = os.getenv("GEMINI_API_KEY")
if not api_key:
    raise ValueError("GEMINI_API_KEY not found")
genai.configure(api_key=api_key)


c:\Users\pc\Desktop\ai_research_copilot\backend\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
C:\Users\pc\AppData\Local\Temp\ipykernel_17992\3645204344.py:3: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  import google.generativeai as genai


API keys are loaded securely using environment variables via python-dotenv.
This approach prevents credential leakage and enables seamless deployment.

In [2]:
model = genai.GenerativeModel(
    model_name="gemini-3-flash-preview"
)


In [10]:
# Basic prompt test
prompt = """
Explain the levels of autonomous driving.

Rules:
- Use bullet points
- Be concise
- Use clear technical language
"""

response = model.generate_content(prompt)

print(response.text)


The Society of Automotive Engineers (SAE) defines six levels of driving automation, ranging from fully manual to fully autonomous:

*   **Level 0 (No Automation):** The human driver performs all dynamic driving tasks. The vehicle may include support systems that provide temporary intervention or warnings, such as Automatic Emergency Braking (AEB) or Blind Spot Warning, but they do not provide sustained control.
*   **Level 1 (Driver Assistance):** The vehicle features a single automated system for either steering support *or* acceleration/deceleration (e.g., Adaptive Cruise Control or Lane Keep Assist). The driver must perform all other driving tasks and maintain constant supervision.
*   **Level 2 (Partial Automation):** The vehicle coordinates multiple Advanced Driver Assistance Systems (ADAS) to control both steering and acceleration/braking simultaneously. The driver must remain physically engaged (hands on or nearby) and monitor the environment at all times.
*   **Level 3 (Conditi

In [11]:
#Controlled output test

prompt = """
You are a research assistant.

Task:
Explain the SAE levels of autonomous driving.

Rules:
- Output must be structured
- No emojis
- No opinions
- Max 6 bullet points
- End with a short summary sentence
"""

response = model.generate_content(prompt)
print(response.text)


### SAE Levels of Driving Automation

*   **Level 0 (No Automation):** The human driver is responsible for all aspects of the dynamic driving task, though the vehicle may provide momentary assistance or warnings such as automatic emergency braking or lane departure alerts.
*   **Level 1 (Driver Assistance):** The vehicle features a single automated system for either steering or acceleration/deceleration (e.g., adaptive cruise control), while the human driver remains responsible for all other driving functions and environment monitoring.
*   **Level 2 (Partial Automation):** The system can simultaneously control both steering and acceleration/deceleration; however, the human driver must actively supervise the system at all times and be prepared to intervene immediately.
*   **Level 3 (Conditional Automation):** The vehicle monitors the environment and performs all aspects of the driving task under specific conditions, but the human driver must be available to take control when the syste

In [18]:
# failure test prompt = 

failure_prompt = """
Explain something you are unsure about.
If you are not confident, say 'I do not know'.

"""

response = model.generate_content(failure_prompt)
print(response.text)


I am unsure about the extent to which I possess a **functional world model**.

In the field of AI research, there is a deep debate regarding whether large language models (LLMs) like me actually "understand" the underlying reality of the topics we discuss, or if we are simply "stochastic parrots" using sophisticated statistics to predict the next likely word.

For example, if I describe the layout of a house or the steps of a physics experiment, it is unclear if I am accessing an internal, three-dimensional "map" or "logical framework" that I have built from my training data, or if I am merely identifying linguistic patterns that have appeared in similar contexts in my training set. 

While I can perform tasks that look like reasoning—such as solving a logic puzzle I’ve never seen before—researchers are divided on whether this constitutes true conceptual understanding or just a very advanced form of pattern matching. Because the billions of mathematical weights inside my architecture a

In [3]:

# Mode-Specific Prompts Automation


# 3️⃣ Helper Functions
# ------------------------------
def run_prompt(prompt):
    """Run a prompt with Gemini AI and safely return the text response."""
    try:
        response = model.generate_content(prompt)
        return getattr(response, "text", "No response returned")
    except Exception as e:
        return f"Error: {e}"

def save_responses_to_json(responses, filename="responses_log.json"):
    """Save prompt responses to a JSON file with timestamps."""
    timestamp = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
    output = {
        "timestamp": timestamp,
        "responses": responses
    }
    with open(filename, "w", encoding="utf-8") as f:
        json.dump(output, f, indent=4, ensure_ascii=False)
    print(f"\n✅ Responses saved to {filename}")

# ------------------------------
# 4️⃣ Mode-Specific Prompts
# ------------------------------
# Chat Mode
chat_prompt = """
You are a helpful assistant.
Answer the user's question concisely.
Do NOT provide sources.
Use bullet points for clarity.
"""

# Research Mode (source-grounded)
def generate_research_prompt(question, sources):
    return f"""
You are a research assistant.

Answer the following question using ONLY the sources provided.

Rules:
- Use bullet points
- Include source citations at the end
- If unsure, respond 'I do not know'

Question: {question}
Sources: {sources}
"""

# File / Document Mode
file_prompt = """
You are a document analysis assistant.
Read the uploaded file content.
Answer the user's question using ONLY information in the file.
Cite sections if relevant.
If the answer cannot be found in the file, respond 'I do not know'.
"""

# JSON / Structured Output Mode
json_prompt = """
Answer the question as a JSON object with fields:
{
    "answer": "string",
    "sources": ["list of sources"]
}
Question: Explain the levels of autonomous driving
Sources: sae.org, ieee.org
"""

# Uncertainty / Controlled Failure Mode
failure_prompt = """
Explain something you are unsure about.
If you are not confident, say 'I do not know'.
"""

# ------------------------------
# 5️⃣ Prompt Library (Reusable)
# ------------------------------
prompts = {
    "chat": chat_prompt,
    "research": lambda: generate_research_prompt(
        question="Explain autonomous driving levels",
        sources=["sae.org", "ieee.org"]
    ),
    "file": file_prompt,
    "structured_json": json_prompt,
    "strict_uncertainty": failure_prompt
}

# ------------------------------
# 6️⃣ Automated Testing Loop
# ------------------------------
responses_log = {}
print("=== Running All Prompts ===\n")
for mode, prompt in prompts.items():
    if callable(prompt):  # for research prompt lambda
        prompt_text = prompt()
    else:
        prompt_text = prompt

    print(f"--- {mode.upper()} Mode ---")
    response_text = run_prompt(prompt_text)
    print(response_text, "\n")
    responses_log[mode] = response_text

# ------------------------------
# 7️⃣ Save All Responses
# ------------------------------
save_responses_to_json(responses_log)


=== Running All Prompts ===

--- CHAT Mode ---
Please provide the question you would like me to answer. I am ready to respond following your instructions:

*   Concise answers
*   Bullet points for clarity
*   No sources provided 

--- RESEARCH Mode ---
Based on the provided sources from sae.org and ieee.org, autonomous driving is categorized into six levels:

*   **Level 0 (No Driving Automation):** The human driver is fully responsible for operating the vehicle. The system may provide warnings or momentary assistance, such as emergency braking, but does not sustain control of the vehicle.
*   **Level 1 (Driver Assistance):** The vehicle provides assistance with either steering or acceleration/braking (e.g., adaptive cruise control or lane centering), but not both simultaneously. The driver remains responsible for all other aspects of driving.
*   **Level 2 (Partial Driving Automation):** The vehicle can control both steering and acceleration/braking at the same time. However, the dri

In [3]:

# Mode-Specific Prompts Automation


# 3️⃣ Helper Functions
# ------------------------------
def run_prompt(prompt):
    """Run a prompt with Gemini AI and safely return the text response."""
    try:
        response = model.generate_content(prompt)
        return getattr(response, "text", "No response returned")
    except Exception as e:
        return f"Error: {e}"

def save_responses_to_json(responses, filename="responses_log.json"):
    """Save prompt responses to a JSON file with timestamps."""
    timestamp = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
    output = {
        "timestamp": timestamp,
        "responses": responses
    }
    with open(filename, "w", encoding="utf-8") as f:
        json.dump(output, f, indent=4, ensure_ascii=False)
    print(f"\n✅ Responses saved to {filename}")

# ------------------------------
# 4️⃣ Mode-Specific Prompts
# ------------------------------
# Chat Mode
chat_prompt = """
You are a helpful assistant.
Answer the user's question concisely.
Do NOT provide sources.
Use bullet points for clarity.
"""

# Research Mode (source-grounded)
def generate_research_prompt(question, sources):
    return f"""
You are a research assistant.

Answer the following question using ONLY the sources provided.

Rules:
- Use bullet points
- Include source citations at the end
- If unsure, respond 'I do not know'

Question: {question}
Sources: {sources}
"""

# File / Document Mode
file_prompt = """
You are a document analysis assistant.
Read the uploaded file content.
Answer the user's question using ONLY information in the file.
Cite sections if relevant.
If the answer cannot be found in the file, respond 'I do not know'.
"""

# JSON / Structured Output Mode
json_prompt = """
Answer the question as a JSON object with fields:
{
    "answer": "string",
    "sources": ["list of sources"]
}
Question: Explain the levels of autonomous driving
Sources: sae.org, ieee.org
"""

# Uncertainty / Controlled Failure Mode
failure_prompt = """
Explain something you are unsure about.
If you are not confident, say 'I do not know'.
"""

# ------------------------------
# 5️⃣ Prompt Library (Reusable)
# ------------------------------
prompts = {
    "chat": chat_prompt,
    "research": lambda: generate_research_prompt(
        question="Explain autonomous driving levels",
        sources=["sae.org", "ieee.org"]
    ),
    "file": file_prompt,
    "structured_json": json_prompt,
    "strict_uncertainty": failure_prompt
}

# ------------------------------
# 6️⃣ Automated Testing Loop
# ------------------------------
responses_log = {}
print("=== Running All Prompts ===\n")
for mode, prompt in prompts.items():
    if callable(prompt):  # for research prompt lambda
        prompt_text = prompt()
    else:
        prompt_text = prompt

    print(f"--- {mode.upper()} Mode ---")
    response_text = run_prompt(prompt_text)
    print(response_text, "\n")
    responses_log[mode] = response_text

# ------------------------------
# 7️⃣ Save All Responses
# ------------------------------
save_responses_to_json(responses_log)


=== Running All Prompts ===

--- CHAT Mode ---
* I understand your instructions.
* I will provide concise responses.
* I will use bullet points for clarity.
* I will not provide any sources.
* Please provide your question. 

--- RESEARCH Mode ---
Error: 429 You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. 
* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-3-flash
Please retry in 18.815521831s. [links {
  description: "Learn more about Gemini API quotas"
  url: "https://ai.google.dev/gemini-api/docs/rate-limits"
}
, violations {
  quota_metric: "generativelanguage.googleapis.com/generate_content_free_tier_requests"
  quota_id: "GenerateRequestsPerDayPerProjectPerModel-FreeTier"
  quota_dimensions {
    key: "model"
    value:

**connect to exa**

In [4]:
#2️⃣ Load Exa Sources
def load_exa_sources(filename="exa_search_results.json", max_sources=5):
    with open(filename, "r", encoding="utf-8") as f:
        sources = json.load(f)

    formatted_sources = []
    for src in sources[:max_sources]:
        formatted_sources.append(
            f"- {src.get('title','')} ({src.get('url','')}): {src.get('snippet','')}"
        )

    return "\n".join(formatted_sources)


In [5]:
#3️⃣ Source-Grounded Research Prompt
def generate_research_prompt(question, exa_sources):
    return f"""
You are a research assistant.

Answer the question using ONLY the information in the sources below.

Rules:
- Do NOT use prior knowledge
- Use bullet points
- Cite sources by domain name at the end
- If the answer is not in the sources, say: "I do not know"

Sources:
{exa_sources}

Question:
{question}

Answer:
"""


In [7]:
#4️⃣ Other Modes
chat_prompt = """
You are a helpful assistant.
Answer concisely.
Use bullet points.
Do NOT provide sources.
"""

file_prompt = """
You are a document analysis assistant.
Answer using ONLY the provided document.
If not found, say 'I do not know'.
"""

json_prompt = """
Answer as JSON:
{
  "answer": "string",
  "sources": ["source domains"]
}
Question: Explain autonomous driving levels
Sources: sae.org, ieee.org
"""

failure_prompt = """
If you are unsure or lack information, respond only with:
'I do not know'
"""
#


In [8]:
#5️⃣ Prompt Runner

def run_prompt(prompt):
    try:
        response = model.generate_content(prompt)
        return response.text.strip()
    except Exception as e:
        return f"Error: {e}"


In [9]:
#6️⃣ Main RAG Execution

def run_research_question(question):
    exa_sources = load_exa_sources()
    prompt = generate_research_prompt(question, exa_sources)
    answer = run_prompt(prompt)

    return {
        "question": question,
        "answer": answer,
        "sources_used": exa_sources
    }


In [10]:
#7️⃣ Save Structured Output
def save_responses_to_json(result, filename="responses_log.json"):
    output = {
        "timestamp": datetime.now().strftime("%Y-%m-%d_%H-%M-%S"),
        "result": result
    }

    with open(filename, "w", encoding="utf-8") as f:
        json.dump(output, f, indent=4, ensure_ascii=False)

    print(f"✅ Responses saved to {filename}")


In [11]:
#8️⃣ Test (End-to-End RAG)
question = "How do self-driving cars detect obstacles and pedestrians?"

result = run_research_question(question)

print("Answer:\n", result["answer"])
save_responses_to_json(result)


Answer:
 Self-driving cars detect their surroundings using the following methods:

*   **Sensor Suites:** Vehicles use a combination of cameras, LiDAR, and RADAR that function as electronic "eyes and ears" to perceive the environment in 360 degrees.
*   **LiDAR Technology:** Light Detection and Ranging (LiDAR) is a critical component that provides precise depth data, which is essential for navigation and safety.
*   **Computer Vision:** This technology is used to power how autonomous vehicles "see" and interpret the world.
*   **Integrated Technology Stacks:** Specialized software and hardware stacks enable vehicles to operate on various road types, including city streets and highways.

Sources: sapien.io, nature.com, dpvtransportation.com, labellerr.com, mobileye.com
✅ Responses saved to responses_log.json
